# Multi-SNR Training — Flow-Matching Speech Enhancement

This notebook trains the Flow-Matching Speech Enhancement model on **multiple SNR levels simultaneously**,
extending the single-SNR=5dB experiments. By exposing the model to a range of noise conditions
(-5, 0, 5, 10, 15 dB), we test whether multi-layer MOSS conditioning helps across diverse SNR levels.

**Experiments**:
1. `none` — no MOSS conditioning (baseline)
2. `last_layer` — MOSS last-layer conditioning
3. `multi_layer` — multi-layer MOSS conditioning with learnable fusion

**Prerequisites**: Run `scripts/extract_multi_snr_features.sh` locally, then pack & upload to Google Drive.

## 0. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# GPU info
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

PROJECT_DIR = "/content/speech-enhancement-project"
# NOTE FOR REVIEWERS: Instead of cloning via GitHub, you can upload the
# submission zip file to Colab and unzip it into the PROJECT_DIR.
# Example: !unzip submission_package.zip -d /content/
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"
DRIVE_BASE = "/content/drive/MyDrive/speech_enhancement_checkpoints"
DRIVE_FEAT = "/content/drive/MyDrive/speech_enhancement_features_multi_snr"

## 1. Clone Repo & Install Dependencies

In [ ]:
import os, subprocess

if not os.path.isdir(PROJECT_DIR):
    subprocess.run([
        'git', 'clone',
        f'https://{GITHUB_TOKEN}@github.com/VictorChen2002/Speech-Enhancement-Project.git',
        PROJECT_DIR
    ], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt 'numpy>=2.0'
print('\nDone. Project at:', os.getcwd())

## 1.5. Login to W&B (optional)

In [ ]:
import wandb
wandb.login()

## 2. Unpack Multi-SNR Features from Drive

Expected archive layout on Drive:
```
speech_enhancement_features_multi_snr/
  features_clean_dac.tar.gz
  features_noisy_dac.tar.gz
  features_moss_last.tar.gz
  features_moss_multi_shard01.tar.gz  (optional shards)
  features_moss_multi_shard02.tar.gz
  ...
```

If you only want to run `none` and `last_layer`, you can skip the moss_multi shards.

In [ ]:
import os, glob, subprocess

os.chdir(PROJECT_DIR)
FEAT_DIR = os.path.join(PROJECT_DIR, 'data', 'features_multi_snr')
os.makedirs(FEAT_DIR, exist_ok=True)

# --- Base features (clean_dac, noisy_dac, moss_last) ---
base_archives = ['features_clean_dac.tar.gz', 'features_noisy_dac.tar.gz', 'features_moss_last.tar.gz']
for archive in base_archives:
    src = os.path.join(DRIVE_FEAT, archive)
    if not os.path.exists(src):
        print(f'  [SKIP] {archive} not found on Drive')
        continue
    subdir = archive.replace('features_', '').replace('.tar.gz', '')
    dest = os.path.join(FEAT_DIR, subdir)
    if os.path.isdir(dest) and len(os.listdir(dest)) > 0:
        print(f'  [OK] {subdir}/ already unpacked ({len(os.listdir(dest))} files)')
        continue
    print(f'  Unpacking {archive} ...')
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['tar', 'xzf', src, '-C', dest], check=True)
    print(f'  -> {len(os.listdir(dest))} files')

# --- MOSS multi-layer shards ---
moss_multi_dir = os.path.join(FEAT_DIR, 'moss_multi')
os.makedirs(moss_multi_dir, exist_ok=True)
shard_pattern = os.path.join(DRIVE_FEAT, 'features_moss_multi_shard*.tar.gz')
shards = sorted(glob.glob(shard_pattern))
if shards:
    existing = len(glob.glob(os.path.join(moss_multi_dir, '*.pt')))
    if existing > 0:
        print(f'  [OK] moss_multi/ already has {existing} files')
    else:
        for shard in shards:
            print(f'  Unpacking {os.path.basename(shard)} ...')
            subprocess.run(['tar', 'xzf', shard, '-C', moss_multi_dir], check=True)
        print(f'  -> {len(glob.glob(os.path.join(moss_multi_dir, "*.pt")))} files total')
else:
    print('  [INFO] No moss_multi shards found — skip multi_layer training or upload them first')

# Verify
for sub in ['clean_dac', 'noisy_dac', 'moss_last', 'moss_multi']:
    d = os.path.join(FEAT_DIR, sub)
    n = len(glob.glob(os.path.join(d, '*.pt'))) if os.path.isdir(d) else 0
    print(f'  {sub:12s}: {n:6d} files')

## 3. Training Configuration

Create a config that points to the multi-SNR feature directory.

In [ ]:
import yaml, torch, os

os.chdir(PROJECT_DIR)

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# --- Override for multi-SNR ---
cfg['data']['features_dir'] = 'data/features_multi_snr'
cfg['data']['split_file'] = 'data/split_multi_snr.json'   # new split for multi-SNR stems

# Adjust batch size by GPU memory
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    if mem_gb > 35:    # A100
        cfg['training']['batch_size'] = 64
    elif mem_gb > 14:  # V100 / T4
        cfg['training']['batch_size'] = 32
    else:
        cfg['training']['batch_size'] = 16
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({mem_gb:.1f} GB) → batch_size={cfg["training"]["batch_size"]}')

# Longer training for more data
cfg['training']['num_epochs'] = 100
cfg['training']['patience'] = 25
cfg['training']['checkpoint_dir'] = 'checkpoints_multi_snr'

CONFIG_PATH = 'configs/colab_multi_snr.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Config saved to {CONFIG_PATH}')
print(f'  features_dir : {cfg["data"]["features_dir"]}')
print(f'  batch_size   : {cfg["training"]["batch_size"]}')
print(f'  num_epochs   : {cfg["training"]["num_epochs"]}')

## 4. Train — No Conditioning (multi-SNR baseline)

In [ ]:
import os, glob, subprocess
os.chdir(PROJECT_DIR)

COND = 'none'
DRIVE_CKPT = os.path.join(DRIVE_BASE, f'multi_snr_{COND}')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Auto-resume: find latest checkpoint
resume_arg = ''
for search_dir in [DRIVE_CKPT, f'checkpoints_multi_snr/{COND}']:
    ckpts = sorted(glob.glob(os.path.join(search_dir, 'step_*.pt')))
    if ckpts:
        resume_arg = f'--resume {ckpts[-1]}'
        print(f'Resuming from {ckpts[-1]}')
        break

if not resume_arg:
    print(f'Starting fresh training for condition={COND}')

cmd = (
    f'python train.py'
    f' --config {CONFIG_PATH}'
    f' --condition_type {COND}'
    f' --wandb --wandb_project speech_enhancement_multi_snr'
    f' --wandb_run_name multi_snr_{COND}'
    f' --drive_ckpt_dir {DRIVE_CKPT}'
    f' {resume_arg}'
)
print(f'CMD: {cmd}')
subprocess.run(cmd.split(), check=True)

## 5. Train — Last-Layer Conditioning (multi-SNR)

In [ ]:
import os, glob, subprocess
os.chdir(PROJECT_DIR)

COND = 'last_layer'
DRIVE_CKPT = os.path.join(DRIVE_BASE, f'multi_snr_{COND}')
os.makedirs(DRIVE_CKPT, exist_ok=True)

resume_arg = ''
for search_dir in [DRIVE_CKPT, f'checkpoints_multi_snr/{COND}']:
    ckpts = sorted(glob.glob(os.path.join(search_dir, 'step_*.pt')))
    if ckpts:
        resume_arg = f'--resume {ckpts[-1]}'
        print(f'Resuming from {ckpts[-1]}')
        break

if not resume_arg:
    print(f'Starting fresh training for condition={COND}')

cmd = (
    f'python train.py'
    f' --config {CONFIG_PATH}'
    f' --condition_type {COND}'
    f' --wandb --wandb_project speech_enhancement_multi_snr'
    f' --wandb_run_name multi_snr_{COND}'
    f' --drive_ckpt_dir {DRIVE_CKPT}'
    f' {resume_arg}'
)
print(f'CMD: {cmd}')
subprocess.run(cmd.split(), check=True)

## 6. Train — Multi-Layer Conditioning (multi-SNR)

Requires `moss_multi` shards to be unpacked in step 2.

In [ ]:
import os, glob, subprocess
os.chdir(PROJECT_DIR)

COND = 'multi_layer'
DRIVE_CKPT = os.path.join(DRIVE_BASE, f'multi_snr_{COND}')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Check moss_multi exists
moss_multi_dir = os.path.join(PROJECT_DIR, 'data', 'features_multi_snr', 'moss_multi')
n_multi = len(glob.glob(os.path.join(moss_multi_dir, '*.pt')))
if n_multi == 0:
    print('ERROR: moss_multi/ is empty — unpack shards in step 2 first!')
else:
    print(f'moss_multi: {n_multi} files')

    resume_arg = ''
    for search_dir in [DRIVE_CKPT, f'checkpoints_multi_snr/{COND}']:
        ckpts = sorted(glob.glob(os.path.join(search_dir, 'step_*.pt')))
        if ckpts:
            resume_arg = f'--resume {ckpts[-1]}'
            print(f'Resuming from {ckpts[-1]}')
            break

    if not resume_arg:
        print(f'Starting fresh training for condition={COND}')

    cmd = (
        f'python train.py'
        f' --config {CONFIG_PATH}'
        f' --condition_type {COND}'
        f' --wandb --wandb_project speech_enhancement_multi_snr'
        f' --wandb_run_name multi_snr_{COND}'
        f' --drive_ckpt_dir {DRIVE_CKPT}'
        f' {resume_arg}'
    )
    print(f'CMD: {cmd}')
    subprocess.run(cmd.split(), check=True)

## 7. Verify Checkpoints on Drive

In [ ]:
import os, glob

for cond in ['none', 'last_layer', 'multi_layer']:
    d = os.path.join(DRIVE_BASE, f'multi_snr_{cond}')
    ckpts = sorted(glob.glob(os.path.join(d, '*.pt')))
    print(f'multi_snr_{cond}: {len(ckpts)} checkpoints')
    if ckpts:
        print(f'  latest: {os.path.basename(ckpts[-1])}')
        best = os.path.join(d, 'best.pt')
        print(f'  best.pt: {"YES" if os.path.exists(best) else "NO"}')

## 8. Evaluate & Compare (multi-SNR)

Run evaluation on the multi-SNR test set.

In [ ]:
import subprocess, os
os.chdir(PROJECT_DIR)

cmd = (
    f'python evaluate.py'
    f' --config {CONFIG_PATH}'
    f' --compare'
    f' --drive_ckpt_dir {DRIVE_BASE}'
)
# Note: evaluate.py --compare looks for <ckpt_dir>/<type>/best.pt
# For multi-SNR checkpoints stored as multi_snr_<type>,
# we need to symlink or pass paths manually.

# Evaluate each condition separately
for cond in ['none', 'last_layer', 'multi_layer']:
    ckpt = os.path.join(DRIVE_BASE, f'multi_snr_{cond}', 'best.pt')
    if not os.path.exists(ckpt):
        print(f'[SKIP] {cond}: no best.pt found')
        continue
    print(f'\n{"="*60}')
    print(f'Evaluating multi_snr_{cond}')
    print(f'{"="*60}')
    cmd = (
        f'python evaluate.py'
        f' --config {CONFIG_PATH}'
        f' --checkpoint {ckpt}'
        f' --condition_type {cond}'
    )
    subprocess.run(cmd.split(), check=True)

## 9. Cross-SNR Evaluation (Generalization Test)

Evaluate the **original SNR=5dB models** on each individual SNR level to measure generalization.

This requires per-SNR feature directories. If you have them, update the paths below.

In [ ]:
import os, json, glob, subprocess, re
import numpy as np
os.chdir(PROJECT_DIR)

# ---- Config ----
SNR_LEVELS = [-5, 0, 5, 10, 15]
CONDITIONS = ['none', 'last_layer', 'multi_layer']
ORIG_CKPT_DIR = '/content/drive/MyDrive/speech_enhancement_checkpoints'
FEAT_MULTI = os.path.join(PROJECT_DIR, 'data', 'features_multi_snr')

# We filter test stems by SNR suffix to evaluate per-SNR
# Stems in multi-SNR features have format: <original_stem>__snr<X>

# Load the split (generated by train.py on first run with multi-SNR features)
split_file = 'data/split_multi_snr.json'
if os.path.exists(split_file):
    with open(split_file) as f:
        split = json.load(f)
    test_stems = split['test']
else:
    # Fallback: discover from clean_dac
    test_stems = sorted([os.path.splitext(f)[0] for f in os.listdir(os.path.join(FEAT_MULTI, 'clean_dac')) if f.endswith('.pt')])
    # Use last 10% as test
    np.random.seed(42)
    np.random.shuffle(test_stems)
    test_stems = test_stems[int(0.9 * len(test_stems)):]

print(f'Total test stems: {len(test_stems)}')

# Group by SNR
snr_groups = {}
for stem in test_stems:
    m = re.search(r'__snr(-?\d+)$', stem)
    if m:
        snr = int(m.group(1))
        snr_groups.setdefault(snr, []).append(stem)

for snr in sorted(snr_groups):
    print(f'  SNR={snr:+3d}dB: {len(snr_groups[snr])} test stems')

## 10. Disconnect Runtime

In [ ]:
# from google.colab import runtime
# runtime.unassign()